# Category Ontology — Build L1 / L2 / L3 layers

In [1]:
import pandas as pd
from openai import OpenAI
from dotenv import load_dotenv
import os
import json

In [ ]:
off_clean = pd.read_csv("off_data_clean2.csv")
usda_clean = pd.read_csv("usda_data_clean2.csv")

In [ ]:
# ── Step 1: Gather all unique categories from both datasets ──────────────
off_cats = off_clean["cat"].value_counts().reset_index()
off_cats.columns = ["original_cat", "count"]
off_cats["source"] = "off"

usda_cats = usda_clean["cat"].value_counts().reset_index()
usda_cats.columns = ["original_cat", "count"]
usda_cats["source"] = "usda"

all_cats = pd.concat([off_cats, usda_cats], ignore_index=True)
print(f"Total unique categories: {all_cats.original_cat.nunique()}")
print(f"  OFF:  {off_cats.shape[0]}")
print(f"  USDA: {usda_cats.shape[0]}")
print(f"  Overlap: {set(off_cats.original_cat) & set(usda_cats.original_cat)}")
all_cats.head()

Total unique categories: 5130
  OFF:  4551
  USDA: 629
  Overlap: {'macaroni and cheese', 'tea bags', 'sausages', 'vegetarian', 'baby foods', 'eggs', 'tomatoes', 'frozen foods', 'snacks variety packs', 'apples', 'grapes', 'snacks', 'melons', 'biscuits/cookies (shelf stable)', 'canned vegetables', 'carrots', 'bananas', 'sweet spreads', 'tofu', 'rice', 'vitamins', 'onions', 'sugar substitutes', 'noodles', 'broccoli', 'strawberries', 'dried fruits', 'beverages', 'popcorn', 'pickled vegetables', 'pears', 'vegetables', 'corn', 'tortillas', 'chickpeas', 'mussels', 'salad dressings', 'bacon', 'frozen vegetables', 'stuffing', 'baking', 'pork', 'salads', 'sauces', 'spreads', 'cereal bars', 'fresh pasta', 'frozen desserts', 'breakfast cereals', 'pineapple'}


,original_cat,count,source
0,breads,7046,off
1,yogurts,5013,off
2,greek-style yogurts,1072,off
3,groceries,631,off
4,bagel breads,436,off


In [ ]:
# ── Step 2: Filter to meaningful categories (>=3 items) ──────────────────
# Singletons/tiny cats are noise — we classify them later via item_name fallback
meaningful_cats = all_cats[all_cats["count"] >= 3].copy()
tiny_cats = all_cats[all_cats["count"] < 3]

print(f"Meaningful cats (>=3 items): {len(meaningful_cats)} covering {meaningful_cats['count'].sum():,} rows")
print(f"Tiny cats (<3 items):        {len(tiny_cats)} covering {tiny_cats['count'].sum():,} rows")
print(f"Coverage: {meaningful_cats['count'].sum() / all_cats['count'].sum() * 100:.1f}%")

Meaningful cats (>=3 items): 2115 covering 1,863,872 rows
Tiny cats (<3 items):        3065 covering 3,697 rows
Coverage: 99.8%


In [ ]:
meaningful_cats.source.value_counts()

source
off     1559
usda     556
Name: count, dtype: int64

In [ ]:
# ── Step 3: Deduplicate — some cats appear in both sources ───────────────
# Group by original_cat, keep total count and all sources
deduped = (
    meaningful_cats
    .groupby("original_cat")
    .agg(total_count=("count", "sum"), sources=("source", lambda x: ",".join(sorted(set(x)))))
    .reset_index()
    .sort_values("total_count", ascending=False)
)
print(f"Unique categories to classify: {len(deduped)}")
deduped.sources.value_counts()

Unique categories to classify: 2075


sources
off         1519
usda         516
off,usda      40
Name: count, dtype: int64

## Phase 2: Define Target Taxonomy & LLM Classification

In [ ]:
# ── Step 4: Define the L1 and L2 super-categories ────────────────────────
# L1: ~20 broad groups (based on USDA SR Legacy + a few additions)
# L2: ~60 specific groups (the retrieval filter level)
# L3: the original category string itself (finest grain)

TAXONOMY = {
    "dairy & eggs": [
        "milk", "yogurt", "cheese", "butter & cream",
        "eggs", "ice cream & frozen yogurt", "dairy desserts"
    ],
    "meat": [
        "beef", "pork", "lamb & game", "processed meats & cold cuts",
        "sausages & hot dogs"
    ],
    "poultry": [
        "chicken", "turkey", "other poultry"
    ],
    "fish & seafood": [
        "fresh fish", "shellfish", "canned & smoked fish",
        "frozen fish & seafood"
    ],
    "fruits": [
        "fresh fruits", "dried fruits", "canned & preserved fruits",
        "fruit juices"
    ],
    "vegetables": [
        "fresh vegetables", "frozen vegetables", "canned vegetables",
        "pickles, olives & peppers", "salads"
    ],
    "grains & pasta": [
        "bread", "pasta & noodles", "rice", "flour & corn meal",
        "cereal", "other grains"
    ],
    "baked goods": [
        "cookies & biscuits", "cakes & pastries", "pies & tarts",
        "crackers", "baking mixes"
    ],
    "snacks": [
        "chips & crisps", "popcorn & puffed snacks", "nuts & seeds",
        "snack bars", "other snacks"
    ],
    "sweets & confectionery": [
        "chocolate", "candy & gummy", "chewing gum & mints",
        "honey & syrups", "jams & spreads", "sugar"
    ],
    "beverages": [
        "water", "soda & soft drinks", "tea", "coffee",
        "energy & sport drinks", "alcoholic beverages",
        "powdered drinks", "other beverages"
    ],
    "condiments & sauces": [
        "ketchup & mustard", "salsa & dips", "salad dressing & mayonnaise",
        "cooking sauces", "seasoning & spices", "vinegar & oils"
    ],
    "fats & oils": [
        "vegetable & cooking oils", "butter & margarine", "other fats"
    ],
    "legumes & beans": [
        "canned beans", "dried legumes", "hummus & bean dips"
    ],
    "soups": [
        "canned soup", "prepared soup", "broth & stock"
    ],
    "prepared & frozen meals": [
        "frozen dinners & entrees", "pizza", "sandwiches & wraps",
        "prepared meals", "meal kits"
    ],
    "baby food": [
        "baby food"
    ],
    "supplements": [
        "dietary supplements", "protein powders", "sport nutrition"
    ],
    "plant-based alternatives": [
        "plant-based milk", "plant-based meat", "plant-based other"
    ],
    "other": [
        "other", "undefined"
    ],
}

# Flatten L2 list for the prompt
L2_LIST = []
L1_FOR_L2 = {}
for l1, l2s in TAXONOMY.items():
    for l2 in l2s:
        L2_LIST.append(l2)
        L1_FOR_L2[l2] = l1

print(f"L1 categories: {len(TAXONOMY)}")
print(f"L2 categories: {len(L2_LIST)}")
print(f"\nL2 list:\n{L2_LIST}")

L1 categories: 20
L2 categories: 87

L2 list:
['milk', 'yogurt', 'cheese', 'butter & cream', 'eggs', 'ice cream & frozen yogurt', 'dairy desserts', 'beef', 'pork', 'lamb & game', 'processed meats & cold cuts', 'sausages & hot dogs', 'chicken', 'turkey', 'other poultry', 'fresh fish', 'shellfish', 'canned & smoked fish', 'frozen fish & seafood', 'fresh fruits', 'dried fruits', 'canned & preserved fruits', 'fruit juices', 'fresh vegetables', 'frozen vegetables', 'canned vegetables', 'pickles, olives & peppers', 'salads', 'bread', 'pasta & noodles', 'rice', 'flour & corn meal', 'cereal', 'other grains', 'cookies & biscuits', 'cakes & pastries', 'pies & tarts', 'crackers', 'baking mixes', 'chips & crisps', 'popcorn & puffed snacks', 'nuts & seeds', 'snack bars', 'other snacks', 'chocolate', 'candy & gummy', 'chewing gum & mints', 'honey & syrups', 'jams & spreads', 'sugar', 'water', 'soda & soft drinks', 'tea', 'coffee', 'energy & sport drinks', 'alcoholic beverages', 'powdered drinks', 'o

In [ ]:
# ── Step 5: LLM batch classification ─────────────────────────────────────
# We send each unique category → LLM → get back one L2 label
# L1 is derived automatically from the L2→L1 mapping
# L3 is the original category itself

import os
import json
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv()  # loads .env from project root

# ── Choose provider / model ──────────────────────────────────────────────
# Groq (free, fast) — uses the OpenAI-compatible endpoint
client = OpenAI()


# Other Groq models you can try:
#   "llama-3.1-8b-instant"        — fastest, less accurate
#   "mixtral-8x7b-32768"          — good middle ground
#   "gemma2-9b-it"                — Google's 9B

# To switch to OpenAI instead, uncomment these two lines:
# client = OpenAI()               # uses OPENAI_API_KEY from .env
# MODEL = "gpt-4o-mini"           # ~$0.05-0.10 for the full batch
# ─────────────────────────────────────────────────────────────────────────

SYSTEM_PROMPT = f"""You are a food classification expert.
Given a food category name, assign it to exactly ONE of these L2 categories:

{json.dumps(L2_LIST)}

RULES:
- Classify by the PRIMARY food type, not by secondary ingredients.
  "peach pie" → "pies & tarts", NOT "fresh fruits"
  "cheese pizza" → "pizza", NOT "cheese"
- Yogurt drinks, flavored milks → "yogurt" or "milk", NOT "other beverages"
- "Frozen X" → classify by X. "frozen fish" → "frozen fish & seafood"
- Protein powders, supplements → "protein powders" or "dietary supplements"
- Breads, buns, bagels, muffins, flatbreads → "bread"
- If ambiguous, prefer the more specific L2 over "other"
- Reply with ONLY the L2 category name, exactly as listed. Nothing else."""


def classify_category(cat_name: str) -> str:
    resp = client.chat.completions.create(
        model="gpt-4.1-mini",
        messages=[
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": cat_name},
        ],
        temperature=0,
        max_tokens=15,
    )
    return resp.choices[0].message.content.strip().lower()


# Test with a few examples before running the full batch
test_cats = ["breads", "greek-style yogurts", "frozen dinners & entrees",
             "potato crisps", "peanut butters", "protein powders", "dark chocolates"]
for tc in test_cats:
    result = classify_category(tc)
    l1 = L1_FOR_L2.get(result, "UNKNOWN")
    print(f"  {tc:40s} → L2: {result:30s} → L1: {l1}")

  breads                                   → L2: bread                          → L1: grains & pasta
  greek-style yogurts                      → L2: yogurt                         → L1: dairy & eggs
  frozen dinners & entrees                 → L2: frozen dinners & entrees       → L1: prepared & frozen meals
  potato crisps                            → L2: chips & crisps                 → L1: snacks
  peanut butters                           → L2: jams & spreads                 → L1: sweets & confectionery
  protein powders                          → L2: protein powders                → L1: supplements
  dark chocolates                          → L2: chocolate                      → L1: sweets & confectionery


In [ ]:
# ── Step 6: Run full batch classification ─────────────────────────────────
# This classifies all meaningful unique categories (~1,600-2,000 strings)
# Cost: ~$0.05-0.10 with gpt-4o-mini

import time

cats_to_classify = deduped["original_cat"].tolist()
print(f"Classifying {len(cats_to_classify)} categories...")

mapping = {}
errors = []

for i, cat in enumerate(cats_to_classify):
    try:
        l2 = classify_category(cat)
        # Validate that the response is in our L2 list
        if l2 not in L2_LIST:
            errors.append((cat, l2, "not in L2 list"))
            l2 = "other"
        mapping[cat] = l2
    except Exception as e:
        errors.append((cat, str(e), "api error"))
        mapping[cat] = "other"

    if (i + 1) % 100 == 0:
        print(f"  {i+1}/{len(cats_to_classify)} done...")

print(f"\nDone! Classified {len(mapping)} categories.")
print(f"Errors/fallbacks: {len(errors)}")
if errors:
    print("Sample errors:")
    for cat, resp, reason in errors[:10]:
        print(f"  {cat} → {resp} ({reason})")

Classifying 2075 categories...
  100/2075 done...
  200/2075 done...
  300/2075 done...
  400/2075 done...
  500/2075 done...
  600/2075 done...
  700/2075 done...
  800/2075 done...
  900/2075 done...
  1000/2075 done...
  1100/2075 done...
  1200/2075 done...
  1300/2075 done...
  1400/2075 done...
  1500/2075 done...
  1600/2075 done...
  1700/2075 done...
  1800/2075 done...
  1900/2075 done...
  2000/2075 done...

Done! Classified 2075 categories.
Errors/fallbacks: 2
Sample errors:
  squid fritter roman-style → fried fish & seafood (not in L2 list)
  pad thai → noodles (not in L2 list)


In [ ]:
# ── Step 7: Save the mapping so we never have to re-classify ──────────────
mapping_df = pd.DataFrame([
    {"original_cat": cat, "cat_l2": l2, "cat_l1": L1_FOR_L2.get(l2, "other")}
    for cat, l2 in mapping.items()
])

mapping_df.to_csv("category_mapping.csv", index=False)
mapping_df.to_json("category_mapping.json", orient="records", indent=2)

print(f"Saved {len(mapping_df)} mappings to category_mapping.csv / .json")
print(f"\nL1 distribution:")
print(mapping_df["cat_l1"].value_counts().to_string())
print(f"\nL2 distribution (top 20):")
print(mapping_df["cat_l2"].value_counts().head(20).to_string())

Saved 2075 mappings to category_mapping.csv / .json

L1 distribution:
cat_l1
dairy & eggs                325
grains & pasta              255
vegetables                  157
baked goods                 152
prepared & frozen meals     151
sweets & confectionery      148
snacks                      142
beverages                   123
meat                        106
other                       100
condiments & sauces          84
fruits                       82
fish & seafood               60
plant-based alternatives     39
poultry                      30
fats & oils                  28
legumes & beans              28
supplements                  27
soups                        25
baby food                    13

L2 distribution (top 20):
cat_l2
cheese                         92
bread                          89
other                          87
yogurt                         86
prepared meals                 67
pasta & noodles                63
fresh vegetables               61
processed m

## Phase 3: Apply Mapping to Both DataFrames

In [ ]:
off_clean = pd.read_csv("off_data_clean2.csv") # insert your right off dataset csv
usda_clean = pd.read_csv("usda_data_clean2.csv") # insert your right usda dataset csv

In [ ]:
# ── Step 8: Apply the mapping to both dataframes ─────────────────────────
# Load mapping (so this cell works even if you restart the kernel)
cat_map = pd.read_csv("category_mapping.csv")
l2_lookup = dict(zip(cat_map["original_cat"], cat_map["cat_l2"]))
l1_lookup = dict(zip(cat_map["original_cat"], cat_map["cat_l1"]))

# Map L1 and L2 onto both dataframes; L3 = original cat
off_clean["cat_l1"] = off_clean["cat"].map(l1_lookup)
off_clean["cat_l2"] = off_clean["cat"].map(l2_lookup)
off_clean["cat_l3"] = off_clean["cat"]  # finest grain

usda_clean["cat_l1"] = usda_clean["cat"].map(l1_lookup)
usda_clean["cat_l2"] = usda_clean["cat"].map(l2_lookup)
usda_clean["cat_l3"] = usda_clean["cat"]

print("=== OFF coverage ===")
print(f"  L1 mapped: {off_clean['cat_l1'].notna().sum():,} / {len(off_clean):,} ({off_clean['cat_l1'].notna().mean()*100:.1f}%)")
print(f"  L2 mapped: {off_clean['cat_l2'].notna().sum():,} / {len(off_clean):,}")
print(f"  L1 unmapped: {off_clean['cat_l1'].isna().sum():,} rows")

print("\n=== USDA coverage ===")
print(f"  L1 mapped: {usda_clean['cat_l1'].notna().sum():,} / {len(usda_clean):,} ({usda_clean['cat_l1'].notna().mean()*100:.1f}%)")
print(f"  L2 mapped: {usda_clean['cat_l2'].notna().sum():,} / {len(usda_clean):,}")
print(f"  L1 unmapped: {usda_clean['cat_l1'].isna().sum():,} rows")

=== OFF coverage ===
  L1 mapped: 37,397 / 40,996 (91.2%)
  L2 mapped: 37,397 / 40,996
  L1 unmapped: 3,599 rows

=== USDA coverage ===
  L1 mapped: 1,826,482 / 1,826,573 (100.0%)
  L2 mapped: 1,826,482 / 1,826,573
  L1 unmapped: 91 rows


In [ ]:
# ── Step 9: Handle unmapped rows (tiny categories) ───────────────────────
# For rows where cat wasn't in our mapping (the <3 item categories),
# classify them by item_name using the same LLM function, or assign "other"

unmapped_off = off_clean[off_clean["cat_l2"].isna()]
unmapped_usda = usda_clean[usda_clean["cat_l2"].isna()]

print(f"Unmapped OFF rows: {len(unmapped_off)} ({len(unmapped_off)/len(off_clean)*100:.1f}%)")
print(f"Unmapped USDA rows: {len(unmapped_usda)} ({len(unmapped_usda)/len(usda_clean)*100:.1f}%)")

# Option A: assign "other" (simple, fast)
off_clean["cat_l1"].fillna("other", inplace=True)
off_clean["cat_l2"].fillna("other", inplace=True)
usda_clean["cat_l1"].fillna("other", inplace=True)
usda_clean["cat_l2"].fillna("other", inplace=True)

print("\nAfter filling unmapped → 'other':")
print(f"  OFF L1 unique: {off_clean['cat_l1'].nunique()}")
print(f"  OFF L2 unique: {off_clean['cat_l2'].nunique()}")
print(f"  USDA L1 unique: {usda_clean['cat_l1'].nunique()}")
print(f"  USDA L2 unique: {usda_clean['cat_l2'].nunique()}")

Unmapped OFF rows: 3599 (8.8%)
Unmapped USDA rows: 91 (0.0%)

After filling unmapped → 'other':
  OFF L1 unique: 20
  OFF L2 unique: 86
  USDA L1 unique: 20
  USDA L2 unique: 86


/var/folders/yz/grwlw1c90b1dbggwlxd8jf200000gn/T/ipykernel_2067/949075583.py:12: ChainedAssignmentError: A value is being set on a copy of a DataFrame or Series through chained assignment using an inplace method.
Such inplace method never works to update the original DataFrame or Series, because the intermediate object on which we are setting values always behaves as a copy (due to Copy-on-Write).

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' instead, to perform the operation inplace on the original object, or try to avoid an inplace operation using 'df[col] = df[col].method(value)'.

See the documentation for a more detailed explanation: https://pandas.pydata.org/pandas-docs/stable/user_guide/copy_on_write.html
  off_clean["cat_l1"].fillna("other", inplace=True)
/var/folders/yz/grwlw1c90b1dbggwlxd8jf200000gn/T/ipykernel_2067/949075583.py:13: ChainedAssignmentError: A value is being set on a copy of a DataFrame or Seri

In [ ]:
# ── Step 10: Validate — spot check a few L1 groups ───────────────────────
combined = pd.concat([off_clean, usda_clean], ignore_index=True)

for l1 in ["dairy & eggs", "fruits", "snacks", "prepared & frozen meals", "beverages"]:
    subset = combined[combined["cat_l1"] == l1]
    sample = subset.sample(min(5, len(subset)), random_state=42)
    print(f"\n{'='*60}")
    print(f"L1: {l1}  ({len(subset):,} rows)")
    print(f"{'='*60}")
    print(sample[["item_name", "cat_l1", "cat_l2", "cat_l3", "source"]].to_string(index=False))


L1: dairy & eggs  (252,840 rows)
                                           item_name       cat_l1                    cat_l2                    cat_l3 source
            southern- style custard, southern- style dairy & eggs            dairy desserts       puddings & custards   usda
          no added sugar pint coffee ice cream, pint dairy & eggs ice cream & frozen yogurt ice cream & frozen yogurt   usda
                      plain whole milk yogurt, plain dairy & eggs                    yogurt                    yogurt   usda
            vanilla greek whole milk yogurt, vanilla dairy & eggs                    yogurt                    yogurt   usda
dierbergs kitchen, holiday dessert spread, cranberry dairy & eggs                    cheese                    cheese   usda

L1: fruits  (53,351 rows)
                                                  item_name cat_l1                    cat_l2                                          cat_l3 source
                       cold-pressed juice

In [ ]:
# ── Step 11: Save final datasets ─────────────────────────────────────────
off_clean.to_csv("off_data_ontology.csv", index=False)
usda_clean.to_csv("usda_data_ontology.csv", index=False)

print(f"Saved off_data_ontology.csv  ({len(off_clean):,} rows)")
print(f"Saved usda_data_ontology.csv ({len(usda_clean):,} rows)")
print(f"\nColumns: {off_clean.columns.tolist()}")

Saved off_data_ontology.csv  (40,996 rows)
Saved usda_data_ontology.csv (1,826,573 rows)

Columns: ['item_name', 'cat', 'kcal_100g', 'fat_100g', 'carbs_100g', 'protein_100g', 'source', 'cat_l1', 'cat_l2', 'cat_l3']


## Finetuning: edit of the mapping and layers

In [ ]:
# Check what landed in "other" — these are likely misclassifications
cat_map = pd.read_csv("category_mapping.csv")
others = cat_map[cat_map["cat_l2"] == "other"]
print(f"Categories mapped to 'other': {len(others)}")
print(others.sort_values("original_cat").to_string(index=False))

Categories mapped to 'other': 87
                                                                       original_cat cat_l2 cat_l1
                                                american indian/alaska native foods  other  other
                                                                         appetizers  other  other
                                                                     baked products  other  other
                                                                      baker's yeast  other  other
                                                                             bakery  other  other
                                                                    bakery products  other  other
                                                                 baking accessories  other  other
                                                        baking additives & extracts  other  other
                                                                       baking needs  

In [ ]:
# Check a specific L2 to see if anything doesn't belong
cat_map[cat_map["cat_l2"] == "bread"].sort_values("original_cat")

,original_cat,cat_l2,cat_l1
169,bagel breads,bread,grains & pasta
469,bagels and english muffins,bread,grains & pasta
424,baguettes,bread,grains & pasta
366,"biscuits, muffins, quick breads",bread,grains & pasta
1544,bran bread,bread,grains & pasta
...,...,...,...
1041,whole-wheat-bread,bread,grains & pasta
441,wholemeal breads,bread,grains & pasta
435,wholemeal sliced breads,bread,grains & pasta
244,yeast breads,bread,grains & pasta


In [ ]:
# Search for a specific food and see where it landed
cat_map[cat_map["original_cat"].str.contains("pizza", case=False)]

## EDA of the cats

In [3]:
usda_ont = pd.read_csv("usda_data_ontology.csv")
off_ont = pd.read_csv("off_data_ontology.csv")

In [4]:
usda_ont.columns

Index(['item_name', 'cat', 'carbs_100g', 'kcal_100g', 'fat_100g',
       'protein_100g', 'source', 'cat_l1', 'cat_l2', 'cat_l3'],
      dtype='str')

In [ ]:
print(usda_ont.cat_l1.unique())
print(off_ont.cat_l1.unique())
#there is nan and "other" in cat_l1, which means some categories were not classified


<StringArray>
[             'fats & oils',      'condiments & sauces',
                    'soups',                    'other',
  'prepared & frozen meals',               'vegetables',
           'grains & pasta',              'baked goods',
                'beverages',                     'meat',
                   'fruits',                   'snacks',
   'sweets & confectionery',             'dairy & eggs',
          'legumes & beans',           'fish & seafood',
                  'poultry',              'supplements',
 'plant-based alternatives',                'baby food',
                        nan]
Length: 21, dtype: str
<StringArray>
[            'dairy & eggs',              'supplements',
           'grains & pasta',                        nan,
                     'meat',                   'snacks',
                   'fruits',  'prepared & frozen meals',
           'fish & seafood',              'baked goods',
                'beverages',   'sweets & confectionery',
        

In [ ]:
df_full = pd.concat([off_ont, usda_ont], ignore_index=True)

In [20]:
df_full[(df_full.cat_l1 == "baked goods") & (df_full.source == "off")].head(20)

,item_name,cat,kcal_100g,fat_100g,carbs_100g,protein_100g,source,cat_l1,cat_l2,cat_l3
353,classic 7'' chocolate cake,cakes,258.000000,5.470000,46.090000,3.910000,off,baked goods,cakes & pastries,cakes
383,belgian milk chocolate rice cakes,puffed rice cakes with milk chocolate,482.352941,20.588235,65.294118,7.647059,off,baked goods,crackers,puffed rice cakes with milk chocolate
888,blackberry pie,pastries,353.982301,15.044248,53.982301,2.654867,off,baked goods,cakes & pastries,pastries
935,2 syrup sponge puddings,pastries,340.952381,9.809524,59.333333,3.523810,off,baked goods,cakes & pastries,pastries
987,buttermilk pancake & waffle mix with candy bits,pancake mixes,350.000000,2.777778,74.074074,9.259259,off,baked goods,baking mixes,pancake mixes
1024,cheddar bunnies,crackers (appetizers),466.666667,20.000000,60.000000,10.000000,off,baked goods,crackers,crackers (appetizers)
1146,"goldfish baked snack crackers, colors cheddar",crackers (appetizers),400.000000,15.000000,56.666667,10.000000,off,baked goods,crackers,crackers (appetizers)
1147,cheddar baked snack crackers,crackers (appetizers),466.666667,16.666667,66.666667,10.000000,off,baked goods,crackers,crackers (appetizers)
1148,goldfish colors cheddar,crackers (appetizers),466.666667,16.666667,66.666667,10.000000,off,baked goods,crackers,crackers (appetizers)
1149,flavor blasted xtra cheddar baked snack crackers,crackers (appetizers),466.666667,16.666667,63.333333,13.333333,off,baked goods,crackers,crackers (appetizers)


In [ ]:
df_full[df_full.cat_l1.isna()]

,item_name,cat,kcal_100g,fat_100g,carbs_100g,protein_100g,source,cat_l1,cat_l2,cat_l3
3,harvest whole wheat bread,chouquettes,232.558140,4.651163,41.860465,11.627907,off,NaN,NaN,chouquettes
6,seriously salt & vinegar,salt and vinegar crisps,512.000000,29.200000,52.000000,6.000000,off,NaN,NaN,salt and vinegar crisps
20,vegan protein - neutral flavour,cooked shrimps,70.588235,1.176471,0.000000,15.294118,off,NaN,NaN,cooked shrimps
112,greek yoghurt,NaN,134.000000,10.100000,4.200000,6.400000,off,NaN,NaN,NaN
141,cavolo nero kale,kale,40.000000,1.600000,1.400000,3.400000,off,NaN,NaN,kale
...,...,...,...,...,...,...,...,...,...,...
1843057,"water, tap",tap water,0.000000,0.000000,0.000000,0.000000,usda,NaN,NaN,tap water
1843058,"water, bottled, plain",bottled water,0.000000,0.000000,0.000000,0.000000,usda,NaN,NaN,bottled water
1843061,"water, enhanced, regular",enhanced water,22.000000,0.000000,5.490000,0.000000,usda,NaN,NaN,enhanced water
1843062,"water, enhanced, diet",enhanced water,0.000000,0.000000,0.000000,0.000000,usda,NaN,NaN,enhanced water


In [ ]:
df_full[df_full.cat_l1 == "other"]
#off seems to perform worse than usda in terms of classification

,item_name,cat,kcal_100g,fat_100g,carbs_100g,protein_100g,source,cat_l1,cat_l2,cat_l3
389,paradise sun,plant-based foods and beverages,32.000000,0.000000,8.000000,0.000000,off,other,other,plant-based foods and beverages
411,oven roasted buffalo style chicken breast,meats,101.694915,2.542373,1.694915,16.949153,off,other,other,meats
442,50/50 blend,fruits and vegetables based foods,24.000000,0.000000,3.530000,2.350000,off,other,other,fruits and vegetables based foods
582,classic white bread,breaded products,269.230769,3.846154,53.846154,7.692308,off,other,other,breaded products
608,beef shaved steak,meats,178.571429,10.714286,0.000000,19.642857,off,other,other,meats
...,...,...,...,...,...,...,...,...,...,...
1867528,"hunts zesty & spicy pasta sauce, 24 oz. can",sauces/spreads/dips/condiments,32.000000,0.000000,6.350000,0.790000,usda,other,other,sauces/spreads/dips/condiments
1867549,"hunts 100% natural tomato ketchup, no high fru...",sauces/spreads/dips/condiments,118.000000,0.000000,29.410000,0.000000,usda,other,other,sauces/spreads/dips/condiments
1867564,"hunt's tomato ketchup, #10 can, 6/114 oz",sauces/spreads/dips/condiments,147.000000,0.000000,29.410000,0.000000,usda,other,other,sauces/spreads/dips/condiments
1867565,"hunts tomato ketchup, 32 oz. squeeze bottle",sauces/spreads/dips/condiments,118.000000,0.000000,29.410000,0.000000,usda,other,other,sauces/spreads/dips/condiments


In [16]:
df_full.cat[(df_full.cat_l1 == "other") & (df_full.source == "usda")].unique()

<StringArray>
[                                                     'sauces/spreads/dips/condiments',
                                  'meat/poultry/other animals  unprepared/unprocessed',
                                                     'pre-packaged fruit & vegetables',
                                                         'baking additives & extracts',
                                                                         'other meats',
                                                                      'crusts & dough',
                                                                      'fish & seafood',
                                                                      'milk additives',
                                                                    'other condiments',
                                                                       'miscellanious',
                                                               'oral hygiene products',
                  

In [17]:
df_full.cat[(df_full.cat_l1.isna()) & (df_full.source == "usda")].unique()

<StringArray>
[                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                             'pork sausages - prepared/processed',
                                                                                                                                                                                                                                                                                                                                                      